In [19]:
import networkx as nx
from scipy.optimize import linear_sum_assignment
from collections import defaultdict

import pandas as pd
import numpy as np

In [20]:
df = pd.read_csv("cassoela - Form.csv")
tracks = np.array([c for c in df.columns if "Strecke" in c])
df_subset = df[["Name"] + list(tracks)]
df_subnp = df_subset.to_numpy()
names = df_subnp[:,0]
preferences = df_subnp[:,1:]



In [21]:
degrees = {
    "Really don't want": 0,
    "Prefer not": 1,
    "Can": 2,
    "Would like": 3,
    "Would really love": 4,
}

weights = np.zeros_like(preferences, dtype=np.int8)
for txt, val in degrees.items():
    weights[preferences == txt] = val

weights

array([[3, 1, 0, 3, 0, 2, 3, 3, 1, 3, 0, 3, 3, 3],
       [2, 3, 1, 3, 2, 3, 2, 2, 2, 3, 1, 2, 2, 2],
       [1, 1, 2, 2, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2],
       [2, 1, 3, 3, 1, 2, 2, 3, 1, 3, 1, 3, 3, 3],
       [2, 0, 1, 4, 0, 0, 3, 2, 0, 3, 0, 4, 4, 4],
       [1, 3, 3, 2, 3, 2, 1, 2, 3, 2, 3, 2, 1, 1],
       [1, 0, 0, 4, 0, 0, 2, 3, 0, 2, 0, 3, 2, 2]], dtype=int8)

In [22]:
matching = linear_sum_assignment(weights, True)
track_runner = {}

for runner_i, track_i in zip(*matching):
    track_runner[names[runner_i]] = tracks[track_i]
    track_runner[tracks[track_i]] = names[runner_i]

In [23]:
for n in names:
    print(f"{n}:", track_runner[n])

Fabiana: Strecke [Strecke 1 - 3.63km - 65hm - Start Bucheggplatz]
Lino: Strecke [Strecke 2 - 13.67km - 160hm - Übergabe Hönggerberg]
Paolo: Strecke [Strecke 5 - 13.92km - 288hm - Übergabe Felsenegg]
Cloé : Strecke [Strecke 3 - 5.88km - 413hm - Übergabe Buchlern]
Mirella: Strecke [Strecke 12 - 6.29km - 78hm - Übergabe Zumikon]
Jaume A. Badia: Strecke [Strecke 9 - 11.13km - 267hm - Übergabe Fluntern]
Lorenzo: Strecke [Strecke 4 - 5.88km - 122hm - Übergabe Uetliberg]


In [24]:
G = nx.graph.Graph()
G.add_nodes_from(names, bipartite=0)
G.add_nodes_from(tracks, bipartite=1)
indices = np.indices(weights.shape)
edges = np.stack([names[indices[0]],tracks[indices[1]], weights])
edges.T.reshape(-1, edges.T.shape[-1])
edges_flatten = edges.T.reshape(-1, edges.T.shape[-1])
G.add_weighted_edges_from(edges_flatten)
matching = nx.max_weight_matching(G)

for track, name in matching:
    track_runner[name] = track
    track_runner[track] = name

In [25]:
for n in names:
    print(f"{n}:", track_runner[n])

Fabiana: Strecke [Strecke 1 - 3.63km - 65hm - Start Bucheggplatz]
Lino: Strecke [Strecke 10 - 8.62km - 159hm - Übergabe Forch]
Paolo: Strecke [Strecke 6 - 10.4km - 234hm - Übergabe Buchlern]
Cloé : Strecke [Strecke 13 - 4.62km - 80hm - Jagdstart Witikon]
Mirella: Strecke [Strecke 14 - 5.59km - 65hm - Übergabe Fluntern]
Jaume A. Badia: Strecke [Strecke 11 - 12.64km - 421hm - Übergabe Egg]
Lorenzo: Strecke [Strecke 4 - 5.88km - 122hm - Übergabe Uetliberg]


In [26]:
any_track_runner = defaultdict(set)
best_matching = linear_sum_assignment(weights, True)
best_score = weights[best_matching].sum()


for run_i, tr_i in zip(*best_matching):
    new_weights = weights.copy()
    new_weights[run_i, tr_i] = 0

    while True:
        track_runner = {}
        matching = linear_sum_assignment(new_weights, True)
        score = new_weights[matching].sum() 
        if score < best_score:
            break

        old_score = score

        for runner_i, track_i in zip(*matching):
            track_runner[names[runner_i]] = tracks[track_i]
            track_runner[tracks[track_i]] = names[runner_i]
            any_track_runner[names[runner_i]].add(tracks[track_i])
            any_track_runner[tracks[track_i]].add(names[runner_i])
        for n in names:
            print(f"{n}:", track_runner[n])

        tr_i_ = np.argmax(matching[1][run_i])
        
        new_weights[run_i, tr_i_] = 0
        break

print(dict(any_track_runner))
    

Fabiana: Strecke [Strecke 7 - 4.51km - 54hm - Übergabe Hönggerberg]
Lino: Strecke [Strecke 2 - 13.67km - 160hm - Übergabe Hönggerberg]
Paolo: Strecke [Strecke 5 - 13.92km - 288hm - Übergabe Felsenegg]
Cloé : Strecke [Strecke 3 - 5.88km - 413hm - Übergabe Buchlern]
Mirella: Strecke [Strecke 12 - 6.29km - 78hm - Übergabe Zumikon]
Jaume A. Badia: Strecke [Strecke 9 - 11.13km - 267hm - Übergabe Fluntern]
Lorenzo: Strecke [Strecke 4 - 5.88km - 122hm - Übergabe Uetliberg]
Fabiana: Strecke [Strecke 1 - 3.63km - 65hm - Start Bucheggplatz]
Lino: Strecke [Strecke 6 - 10.4km - 234hm - Übergabe Buchlern]
Paolo: Strecke [Strecke 5 - 13.92km - 288hm - Übergabe Felsenegg]
Cloé : Strecke [Strecke 3 - 5.88km - 413hm - Übergabe Buchlern]
Mirella: Strecke [Strecke 12 - 6.29km - 78hm - Übergabe Zumikon]
Jaume A. Badia: Strecke [Strecke 2 - 13.67km - 160hm - Übergabe Hönggerberg]
Lorenzo: Strecke [Strecke 4 - 5.88km - 122hm - Übergabe Uetliberg]
Fabiana: Strecke [Strecke 1 - 3.63km - 65hm - Start Bucheggpl